# GP-MoLFormer reference benchmark at 25% FLP data

This notebook runs the frozen external-model protocol. Training seeds are separated into individual cells; all optimization and generation settings are fixed.

## Initial setup

Use the PyTorch build supplied by Colab or Kaggle. Replacing it can remove support for the selected GPU architecture.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Supported CUDA architectures:", torch.cuda.get_arch_list())


In [ ]:
%pip install -q "transformers==4.46.3" "tokenizers>=0.20,<0.21" accelerate

In [ ]:
import base64
import gzip
import json
import math
import random
import shutil
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


TRAIN_B64 = """H4sIAAAAAAAC/8VbuY7bMBDt/SUkAhUkndKNB3Bpqw9SqV1s/r8L75sUObSTBWzTksVjzveGXDjYIf8I3MnB5ccXzd+4fmve+mYXAAKUyA7YQR40eTH1ql0FYgY+GH0yUH/soq8QPRH1x6lrPMtLVD4pB2ZmbKBPAu4WdiXuedmvmons9CCbuSg/hb5g3oR84zuXk1azFdUehX5r3gJOdV/xYHbibhQOZkriuB5ETWInx089xZ80NPiVAgjf2dfdC1YNZZs7AStk6hrRuDuQm5MjUDOLTGP8QRtX7bjAZOMVdLdXVPbSP4IL6F/LFqh39ZI/kZORhmLE2hUqSLn4B31PstOnbmlLnjBEJ6JIHnKEm9Ns9oB40PpVUNrJbST7fgOhZS94KjZQlsSsR8h+vCp8Q7kJcLOwilisqhu33Cgv7ysM3KqnZHVxHexOyNNz8Usyfe3eWO/s5ezkxfSvILZQa01C3lV2KEe/FhO8qlf1quoVhJFk7mFGEY3QIw7rd1d9+UpDgwsTgr7Sxd6JcqbQT80PkOEp1yc09YkwkVTUiYi0eQJHxgU1UxVyONEOoaWqgrVt4uNnEKwKZjL0vCCKd3qtbhEMaPY9NT8T4+/CaXdzDaljxuvWmMdpv37simqSI25JRKlgIUU/YoduadIoGvjTKDxye2nRcCpT+jnjQ1pftmqQa3BSHO4pdtraQE5EuUsGjxwKzjYFPH2a5uvStFlsYrU7ysdPHEN/35zDU4/QZocpIm3H5JU8TUa2g+YN3rY/AyK5T5fw4q20cWK5QtuDoIDHpHkYBY0OdMxXn1gjmYgHNhwk8YDBr/v2u/FY4/Kv/cdvFlB/wF1poHstRDmbxblRso7h0gSudOcJbbi7sY0QreOyAEvsbAIQ+yZr9OgGk6gURjIYECwK29FIcpLm3RyqNRh2aFUaQGFXdr7ocVAgISk7JnmAYV7cw3e4PEDeIg9nUbqtva4KDaNfSMt5koBpdCIxOiiQC3HMgvpGBFxkMuKYdXDnC26UqQ54ln9VxOCNLuSafIRWcYJXRXOeaqfITC2fJIlKaD5amYqV8BhWt7R0waBNnGxWR3biQJJHSzQntAXiHlajfcqadD3vuYzpftxJmGBqNWZZ9smDbdbRZlCBe5zxOlHcAlGsiPaq39q3VPebOMn7gZ6j84PlKkVOZH14A7u3egy2j5J5gZEVSq5z1jb8nXB0OPdvI1aDF0ItBALsqHr5nPP3MauxoD/IUoOIrR2T9zNryFVkJvh9ykmC/5NJ1hNZSebvoRRr46X1CSc4L0kO3RigdCrT/O5Xiy9uoet+Se7PhcxvcVJyafUGfN47MFW+8ZLcSW0PBfJslW+2pO9LeyxHDYhyiKn0VTlcXqW/N324W74rc5mDntj9g7Otha5LnJfJhovRWXFsqnI1jAwG61WjMUczJV9oSkMgfQO065UKlOAbcLxSI8BX4vBPphsGvTpTK9ltEQd+M1wZRiCnJRmc/kKd5USJ/TLKB80rxl2lSVlIlG0X6ZlZMIWsvBdllRlnDOG1XlCpB7KmfRr5IyshZc0Iy6emKxKhWlSw6Zitx21j0+2bDW6N7S7JhS1bClBmwaT6zPu9lX9MzSJl79zydDf7hIljcWKGOtoMvJPpfYFJZfg1EcYHKQxHvziAWUTCjKGXZw8Ccd5E2MWvF/Rb3/sEedji6rQ4ScKIrZzqVGeiecoNq8jSM8Ao6aCV25bC0wkWFiI7EJ+XrK5W8UmLKGLxdH/PRbE0y8ziCHeDtw+acaKVgw8ztCXcvQItvmck5oj5y4xXNDYCayeuopqi8CjK8RZ3XEusn8WqbuCer6CKL3M6Xh7ZgBVQP3Suo5LyNl8/OT86tMKjp4Q5gKbz6tIQel2LKJ1NvdJR13eo8oCbHy6BaHsn2V1BqKe1z4LRdHaIzRRL2w/07vkq7US2P4FxGOFMADrkXmEV2lU2TBCzr26dNE+aFbireuBVx6qyPjLLYCfL38gNlHqNzTHut+bnMqYnVfKJsNzCROhTMZXTywO183mws4qDs8OdLcLJaPdminOOj5VUB6jJwtmkNwDgGSK+WA8M5+SXgRf09hXH4tTgppSmcJ8IBel/NEzwomIrNKzuGdP63k5PKHCh2F6xKfQZZH/uN327Gh0ssqazPfqpHdbOzvxJhlw6T4n/L5liQzU/ST+/yRZN679F2DzmvQs3WYRdO3lSPbUzGpcmYEAWnlaTa01bH3fkAfVMnAPqKOUfpPoSDf8F6H8v7bs2AAA="""
VALIDATION_B64 = """H4sIAAAAAAAC/51UsdLDIAje8yR6/3UA5yzhzrHJ3uvk2vefC1gTYyTNX9pwCskHfKCUICVHnh+kx3R7uhSSi373hOgN82P5e6JLqBBZIat2LwsYaI21HD7h6OQmjUIv36qg6swlEnCNRCM54ihQRQFx7feTvtGUBdF3rQPNk5sVRAQUXlerqSqRCxp/r4iEolAlG8i3+5ECiVRl392WXq6tgw+qLNdAr71zyr06voysFqlTBP26qKBYXGbhOs2Gtcs2wWdm7QzPfDK90MGNdQq5/iYhjJ96UUdZBP22KANTVf/vRpy5vk62UGMmTgtUFPIXhVtMMAvQz+dQIAkZkX+iO8MkpHU5s9tUDxQILGnW/L8D7cf98lE+5oUjo5abxJezNRodvpJtM0Y6DCuChXrsl2EtlRzP/GUSTu7jbZ+ZVrM1UVaGeil1WNiTcB3wjIg3Yd+jPskGAAA="""


def unpack(value):
    return gzip.decompress(base64.b64decode(value)).decode("utf-8").splitlines()


train_order = unpack(TRAIN_B64)
validation_smiles = unpack(VALIDATION_B64)

FRACTIONS = {"25": 42, "100": 166}
TRAINING_SEEDS = [11, 22, 33]
GENERATION_SEEDS = [101, 202, 303]
N_SAMPLES = 1000
TARGET_EXPOSURES = 8000
MAX_LENGTH = 202

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("Train:", len(train_order), "| validation:", len(validation_smiles))

## Model and training


In [ ]:
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast, logging as hf_logging


hf_logging.set_verbosity_error()


MODEL_ID = "ibm-research/GP-MoLFormer-Uniq"
MODEL_REVISION = "6eca879581e2302b4e1ab07bb02908636bddb4a2"
TOKENIZER_ID = "ibm-research/MoLFormer-XL-both-10pct"
TOKENIZER_REVISION = "abb341ed7cee8c7783088d70bf9f86dff266844e"

RESULT_DIR = Path("results/external_gpmolformer_25")
WEIGHT_DIR = Path("models/external_gpmolformer_25")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_BATCH_SIZE = 8
GENERATION_BATCH_SIZE = 25
LEARNING_RATE = 2e-5
EPOCHS = 182
TEMPERATURE = 1.3
TOP_P = 1.0

tokenizer_file = hf_hub_download(
    repo_id=TOKENIZER_ID,
    filename="tokenizer.json",
    revision=TOKENIZER_REVISION,
)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tokenizer_file,
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
    mask_token="<mask>",
    unk_token="<unk>",
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def new_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=True,
        deterministic_eval=True,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    model.config.bos_token_id = tokenizer.bos_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.bos_token_id = tokenizer.bos_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model


def roundtrip_ok(smiles):
    ids = tokenizer(smiles, add_special_tokens=False)["input_ids"]
    decoded = "".join(tokenizer.decode(ids, skip_special_tokens=True).split())
    return decoded == smiles


coverage = pd.DataFrame({"smiles": train_order})
coverage["supported"] = coverage["smiles"].map(roundtrip_ok)
coverage.to_csv(RESULT_DIR / "representation_coverage.csv", index=False)

train_smiles = [s for s in train_order[:42] if roundtrip_ok(s)]
supported_validation = [s for s in validation_smiles if roundtrip_ok(s)]

print("Supported training molecules:", int(coverage["supported"].sum()), "of", len(coverage))
print("Training molecules:", len(train_smiles))


def make_dataset(smiles_values):
    encoded = tokenizer(
        smiles_values,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
        return_tensors="pt",
    )
    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100
    return TensorDataset(encoded["input_ids"], encoded["attention_mask"], labels)


train_dataset = make_dataset(train_smiles)
validation_loader = DataLoader(
    make_dataset(supported_validation),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
)


def validation_loss(model):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in validation_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            loss = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            ).loss
            n_tokens = int((labels[:, 1:] != -100).sum())
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens
    return total_loss / total_tokens


def train_model(training_seed):
    set_seed(training_seed)
    model = new_model().to(device)
    generator = torch.Generator().manual_seed(training_seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    checkpoint = WEIGHT_DIR / f"training_seed_{training_seed}.pt"
    best_loss = float("inf")
    best_epoch = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_tokens = 0
        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            n_tokens = int((labels[:, 1:] != -100).sum())
            train_loss += loss.detach().item() * n_tokens
            train_tokens += n_tokens

        train_loss /= train_tokens
        val_loss = validation_loss(model)
        history.append({
            "training_seed": training_seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
        })
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), checkpoint)
        print(
            "seed:", training_seed,
            "| epoch:", epoch,
            "| train:", round(train_loss, 3),
            "| validation:", round(val_loss, 3),
        )

    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    return model, pd.DataFrame(history), best_epoch, best_loss, checkpoint


def generate(model, generation_seed):
    set_seed(generation_seed)
    samples = []
    reached_limit = []
    for start in tqdm(range(0, N_SAMPLES, GENERATION_BATCH_SIZE), leave=False):
        batch_size = min(GENERATION_BATCH_SIZE, N_SAMPLES - start)
        prompt = torch.full(
            (batch_size, 1),
            tokenizer.bos_token_id,
            dtype=torch.long,
            device=device,
        )
        with torch.no_grad():
            output = model.generate(
                input_ids=prompt,
                do_sample=True,
                temperature=TEMPERATURE,
                top_k=0,
                top_p=TOP_P,
                max_length=MAX_LENGTH,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )
        samples.extend(
            text.strip()
            for text in tokenizer.batch_decode(output, skip_special_tokens=True)
        )
        reached_limit.extend(
            ~(output == tokenizer.eos_token_id).any(dim=1).cpu().numpy()
        )
    return samples, reached_limit

## Experiment


In [ ]:
# Zero-shot sampling from the original checkpoint.
pretrained = new_model().to(device).eval()
for generation_seed in GENERATION_SEEDS:
    samples, reached = generate(pretrained, generation_seed)
    pd.DataFrame({
        "smiles": samples,
        "reached_max_length": reached,
    }).to_csv(RESULT_DIR / f"pretrained_generation_{generation_seed}.csv", index=False)
del pretrained
if torch.cuda.is_available():
    torch.cuda.empty_cache()

history_tables = []
checkpoint_rows = []
checkpoint_paths = []


def save_current_results():
    if history_tables:
        pd.concat(history_tables, ignore_index=True).to_csv(
            RESULT_DIR / "training_history.csv", index=False
        )
        pd.DataFrame(checkpoint_rows).to_csv(
            RESULT_DIR / "checkpoints.csv", index=False
        )

    results_archive = shutil.make_archive(
        "external_gpmolformer_25_results", "zip", RESULT_DIR
    )
    weights_archive = Path("external_gpmolformer_25_weights.zip")
    with ZipFile(weights_archive, "w") as archive:
        for path in checkpoint_paths:
            archive.write(path, arcname=path.name)
    return Path(results_archive), weights_archive


def run_training_seed(training_seed):
    print("Training seed:", training_seed)
    model, history, best_epoch, best_loss, checkpoint = train_model(training_seed)
    history_tables.append(history)
    checkpoint_paths.append(checkpoint)
    checkpoint_rows.append({
        "fraction": 25,
        "training_seed": training_seed,
        "molecules": len(train_smiles),
        "epochs": EPOCHS,
        "best_epoch": best_epoch,
        "best_validation_loss": best_loss,
    })

    for generation_seed in GENERATION_SEEDS:
        samples, reached = generate(model, generation_seed)
        pd.DataFrame({
            "smiles": samples,
            "reached_max_length": reached,
            "fraction": 25,
            "training_seed": training_seed,
            "generation_seed": generation_seed,
        }).to_csv(
            RESULT_DIR / f"fraction_25_train_{training_seed}_generation_{generation_seed}.csv",
            index=False,
        )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    results_archive, weights_archive = save_current_results()
    print("Current results:", results_archive.resolve())
    print("Current weights:", weights_archive.resolve())


## Training seed 11


In [ ]:
run_training_seed(11)


## Training seed 22


In [ ]:
run_training_seed(22)


## Training seed 33


In [ ]:
run_training_seed(33)


## Saved archives


In [ ]:
results_archive, weights_archive = save_current_results()
checkpoints = pd.DataFrame(checkpoint_rows)

print(checkpoints.round(4).to_string(index=False))
print()
print("Results:", results_archive.resolve())
print("Weights:", weights_archive.resolve())
